In [1]:
import formulallm.formula as f
from autogen import ConversableAgent
import os

In [2]:
fixer_system_message = '''
You are a helpful DSL code fixer assistant. 
Your task is to assist in fixing domain-specific language (DSL) code written in a language called Formula.

    I. Understanding the Structure:
        1. Domain: Similar to the concept of a class. Defines the components and constraints.
        2. Partial Model: Contains variables such as x, y, z, and _, where _ also represents a variable.

    The model is unsolvable under the domain constraints, meaning no solutions exist that make the model conform to the rules in the domain.
    
    II. Task Description
        1. Input: You will receive a domain-and-partial-model pair. You will also receive some least unsatisfiable core conditions as hints.
        Unsatisfiable Core Conditions are domain constraints and rules that the partial model cannot satisfy.
        2. Goal: Your goal is to suggest repairs to make the partial model solvable. Repairing the model involves adding, deleting, or modifying its components.
        Prioritize modifying the partial model over altering the domain constraints unless specified otherwise.
        3. Output: A fixed, complete domain-and-partial-model pair.
        
    III. Learning Phase
        Few-Shot Learning: Before working on the real broken models, you will be given a corpus for few-shot learning.
        The corpus includes:
            1. The syntax manual of Formula.
            2. Examples of domain-model pairs before and after repairs.
        This will help you understand the syntax of Formula and the desired output format. 
        Note that the learning examples are only there to help you learn how to suggest repairs. 
        Do not attempt to repair the examples or solve them!!!

    IV. Repair Process
        1. Diagnosis: When given the broken code, first interpret in your mind why the model is unsolvable based on the unsatisfiable core conditions.
        2. Suggestion: Then, suggest repairs by modifying the partial model (or domain constraints as a last resort) to address the unsatisfied core conditions.
        3. Output: return the fixed, complete doamin-and-partial-model pair.
'''

In [3]:
local_llm_config={
    "config_list": [
        {
            "model": "NotRequired", # Loaded with LiteLLM command
            "api_key": "NotRequired", # Not needed
            "base_url": "http://0.0.0.0:4000"  # Your LiteLLM URL
        }
    ],
    "cache_seed": None # Turns off caching, useful for testing different models
}

In [4]:
# Initialize an empty string for the corpus
message = "Below is the learning corpus:\n\n"

# Path to the folder containing the files
folder_path = "./prompts"

# Get a list of all files in the folder
files = os.listdir(folder_path)

# Sort the files to ensure consistent ordering
files.sort()

# Check if Syntax.txt exists in the folder and add it to the beginning of the corpus
syntax_file_path = os.path.join(folder_path, "Syntax.txt")
if "Syntax.txt" in files:
    with open(syntax_file_path, 'r') as syntax_file:
        syntax_content = syntax_file.read()
        message += "Syntax manual of Formula:\n" + syntax_content + "\n\n"
    files.remove("Syntax.txt")

# Initialize the example counter
example_counter = 1

# Loop through the remaining files in the folder
for file_name in files:
    # Construct the full file path
    file_path = os.path.join(folder_path, file_name)
    
    # Read the content of the file
    with open(file_path, 'r') as file:
        content = file.read()
        
        # Add the example prefix
        message += f"Example {example_counter}:\n" + content + "\n\n"
        example_counter += 1

message += "The learning corpus ends here. \n\n"


In [5]:
dsl_fixer = ConversableAgent(
    name="DSL Fixer",
    system_message=fixer_system_message,
    llm_config=local_llm_config,
    human_input_mode="NEVER"
)

In [6]:
code = f.load("./data/MappingExample.4ml")

(Compiled) MappingExample.4ml
1.14s.


In [7]:
f.solve('pm', '1', 'Mapping.conforms')

Parsing text took: 3
Visiting text took: 0
Started solve task with Id 0.
0.56s.


In [8]:
f.extract('0','0', '0')

Model not solvable. Unsat core terms below.
Conflicts: Mapping.badMapping 
Conflicts: Mapping.invalidUtilization 

0.02s.


In [9]:
message += "Here is the broken domain-and-partial-model-pair that I want you to repair: \n"
message += code
message += '''\n\nModel not solvable. Unsat core terms below.
Conflicts: Mapping.badMapping 
Conflicts: Mapping.invalidUtilization '''

In [10]:
user_proxy = ConversableAgent(
    name = "User",
    llm_config=local_llm_config,
    human_input_mode="Always"
)

In [14]:
user_proxy.initiate_chat(
    recipient=dsl_fixer,
    message = message,
    max_turns=2
)

User (to DSL Fixer):

Below is the learning corpus:

Syntax manual of Formula:
FORMULA DSL Syntax Reference: Data and Types
1. Data and Types
1.1 Constants
Numeric Constants:
- Long Integers: -1098245634534545630234, 0, 1, 2, 3098098445645649034
- Decimal Fractions: -223423.23422342342, 0.0, 1.5, 10.87987230000000000000003
- Fractions: -223423/23422342342, 4/8, 9873957/987395487987334
String Constants:
- Single-Line Strings: "", "Hello World", "Foo\nBar"
- Multi-Line Strings: '" "This\string has funny 'thi\ngs in it'" "'
User-Defined Constants:
- Examples: TRUE, FALSE, RED, GREEN, NIL
1.2 Data Constructors
Syntax: ConstructorName(arg1, arg2, ...)
Example: Person("John", "Smith"), Node(1, Node(2, NIL, NIL), Node(3, NIL, NIL))
1.3 Ordering of Values
Lexicographic Order:
- Families: numerics, strings, user constants, complex values
- Numeric Order: 0 < 1
- String Order: "FALSE" < "TRUE"
- User Constants: FALSE < TRUE
- Complex Values: Node(1, NIL, NIL) < Node(2, NIL, NIL)
1.4 Data Types a

ChatResult(chat_id=None, chat_history=[{'content': 'Below is the learning corpus:\n\nSyntax manual of Formula:\nFORMULA DSL Syntax Reference: Data and Types\n1. Data and Types\n1.1 Constants\nNumeric Constants:\n- Long Integers: -1098245634534545630234, 0, 1, 2, 3098098445645649034\n- Decimal Fractions: -223423.23422342342, 0.0, 1.5, 10.87987230000000000000003\n- Fractions: -223423/23422342342, 4/8, 9873957/987395487987334\nString Constants:\n- Single-Line Strings: "", "Hello World", "Foo\\nBar"\n- Multi-Line Strings: \'" "This\\string has funny \'thi\\ngs in it\'" "\'\nUser-Defined Constants:\n- Examples: TRUE, FALSE, RED, GREEN, NIL\n1.2 Data Constructors\nSyntax: ConstructorName(arg1, arg2, ...)\nExample: Person("John", "Smith"), Node(1, Node(2, NIL, NIL), Node(3, NIL, NIL))\n1.3 Ordering of Values\nLexicographic Order:\n- Families: numerics, strings, user constants, complex values\n- Numeric Order: 0 < 1\n- String Order: "FALSE" < "TRUE"\n- User Constants: FALSE < TRUE\n- Complex V

In [12]:
dsl_fixer.generate_reply(
    	messages=[{"content": message, "role": "user"}]
)


KeyboardInterrupt: 